In [1]:
import yaml
import os
import shutil
import subprocess
import glob
import numpy as np

In [2]:
#Function creates a list of coordinates
def create_coordinates(x):
    
    longitude = np.arange(-180,180,x)
    latitude = np.arange(-80,80,x)

    coordinates = np.column_stack([np.repeat(longitude,len(latitude)),
                                   np.tile(latitude, len(longitude)),
                                   np.repeat(longitude,len(latitude))+x,
                                   np.tile(latitude, len(longitude))+x])
            
    return coordinates

In [ ]:
#Function creates a folder of configuration files based on a template and an output folder for simulation output
def configure_experiment(template, coordinates, exp_name):

    #Loading in configuration template
    with open(template, 'r') as f:
        template = yaml.load(f, Loader=yaml.FullLoader)
    
    output_folder = f'/glade/derecho/scratch/considine/{exp_name}'
    config_folder = f'/glade/u/home/considine/{exp_name}_config'

    #Creating configuration and output folder
    x = True
    count = 0

    while x == True:
    
        try: 
            os.mkdir(config_folder)
            os.mkdir(output_folder)
        except Exception as e:
            print(f"An error occurred: {e}")
            count += 1
            config_folder = f"{config_folder}{str(count)}"
            output_folder = f"{output_folder}{str(count)}"
            continue

        x = False

    print(f"Output folder created: {output_folder}")
    
    #Creating a configuration file for each set of coordinates
    for c in coordinates: 

        template['tagging_region'] = c
        
        if c[0] < 0:
            coord_name = f'_n{str(abs(c[0])).zfill(3)}_'
        else:
            coord_name = f'_{str(c[0]).zfill(4)}_'
        if c[1] < 0:
            coord_name = f'{coord_name}n{str(abs(c[1])).zfill(3)}'
        else:
            coord_name = f'{coord_name}{str(c[1]).zfill(4)}'
            
        template['output_folder'] = f'{output_folder}/output{coord_name}'
        
        #Writing configuration file for this specific set of coordinates
        file_name = f'config{coord_name}.yaml'
    
        with open(file_name, "a") as f:
            file_contents = yaml.dump(template)
            f.write(file_contents)

        shutil.move(file_name, config_folder) #Moving file into configuration folder


In [ ]:
#Runs a simulation for each configuration file in a configuration folder
def run_experiment(config_folder):

    derecho_job = ['#!/bin/bash', 
             '#PBS -A WYOM0161', 
             '#PBS -l walltime=07:00:00', 
             '#PBS -q main', 
             '#PBS -l select=1:ncpus=1:mem=4GB', 
             '#PBS -N ', 
             '#PBS -e ', 
             '#PBS -o ', 
             'module load conda', 
             'conda activate wamenv', 
             'wam2layers track '] 
    
    #Grabbing every configuration file
    for file in glob.glob(f'{config_folder}/c*'): 
    
        job_name = 'job'+ file[6:16]
       
        derecho_job[5] = f'#PBS -N {job_name}'
        derecho_job[6] = f'#PBS -e {job_name}_e.txt'
        derecho_job[7] = f'#PBS -o {job_name}_o.txt'
        derecho_job[10] = f'wam2layers track {config_folder}/{file}'

        with open(f'{job_name}.sh',"w") as f:
            f.write('\n'.join(derecho_job))

        subprocess.call(f'qsub {job_name}.sh',shell=True)

In [ ]:
coordinates = create_coordinates(4)
configure_experiment('2011-2012.yaml',coordinates[0:1],'lmao')
run_experiment('/glade/derecho/scratch/considine/lmao')